# Clustering

## K-Means

Instances which are more similar to each other are given the same _label_. It works by finding centroids of each "blob"'s center and assign the instances to the closest blob. 

- **Hard Clustering**: each instance is assigned to a single cluster. 
- **Soft Clustering**: each instance is assigned a score (it can be distance between instance and centroid; or it can be similarity score such as Gaussian Radial Basis Function (RBF)).

It works by initializing random centroids, generating labels with those centroids, then update the centroids and generate labels, and so on, until the centroids stop moving. The algorithm is guaranteed to converge in a finite number of steps, as mean squared distance between instances and closest centroid can only go down at each step.  

**Inertia**: mean squared distance between each instance and its closest centroid. It is used as a performance metric. 

In Scikit-Learn, it is implemented using `sklearn.cluster.KMeans` class.

### Improvements to K-Means Clustering

1. **K-Means++**: K-Means with a smarter initialization step that tends to select centroids that are distant from one another. This is default method in Scikit-Learn's `KMeans()` class.
2. **Accelerated K-Means**: accelerates algorithm by exploiting the triangle inequality and by keeping track of lower and upper bounds for disrtances between instances and centroids. Used by default in Scikit-Learn.
3. **Mini-batch K-Means**: instead of full dataset at each iteration, it uses mini-batches to move the centroids slightly in each iteration. Can be used with `sklearn.cluster.MiniBatchKMeans` class. 

### Finding the optimal number of clusters

#### Silhouette Score

- Silhouette score is the mean silhouette coefficient over all instances. 
- An instance's silhouette coefficient is $\frac{(b-a)}{\text{max}(a, b)}$ where $a$ is mean distance to other instances in the same cluster and $b$ is the mean distance to the instances of the next closest cluster, defined as the one that minimizes $b$.
- The coefficient varies between -1 to 1.
- Closer to +1 means instance is inside it's own cluster, and far from others.
- Closer to 0 means it's close to a cluster boundary.
- Closer to -1 means it's assigned to the wrong cluster.
- In Scikit-Learn, it is implemented via `sklearn.metrics.silhouette_score` class.

### Limits of K-Means

- It needs to run several time to avoid suboptimal solutions.
- The number of clusters ($k$) must be specified.
- Behaves differently when clusters are of different sizes, densities, or if they are of nonspherical shapes.

### Applications

#### Using Clustering for Image Segmentation

In [6]:
from matplotlib.image import imread
import os

image = imread(os.path.join("images", "ladybug.jpeg"))
image.shape

(452, 679, 3)

In [7]:
X = image.reshape(-1, 3)

In [8]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=8).fit(X)
segmented_img = kmeans.cluster_centers_[kmeans.labels_]
segmented_img = segmented_img.reshape(image.shape)

#### Using Clustering from Preprocessing

In [14]:
from sklearn.datasets import load_digits

X_digits, y_digits = load_digits(return_X_y=True)

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_digits, y_digits)

In [16]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression()
log_reg.fit(X_train, y_train)

/home/haider/anaconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [17]:
log_reg.score(X_test, y_test)

0.9755555555555555

In [18]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("kmeans", KMeans(n_clusters=50)), 
    ("log_reg", LogisticRegression()), 
])
pipeline.fit(X_train, y_train)

/home/haider/anaconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,steps,"[('kmeans', ...), ('log_reg', ...)]"
,transform_input,None
,memory,None
,verbose,False
,n_clusters,50
,init,'k-means++'
,n_init,'auto'
,max_iter,300
,tol,0.0001
,verbose,0
,random_state,None


In [19]:
pipeline.score(X_test, y_test)

0.9666666666666667

In [20]:
from sklearn.model_selection import GridSearchCV

param_grid = dict(kmeans__n_clusters=range(2, 100))
grid_clf = GridSearchCV(pipeline, param_grid, cv=3,  verbose=2)
grid_clf.fit(X_train, y_train)

Fitting 3 folds for each of 98 candidates, totalling 294 fits
[CV] END ...............................kmeans__n_clusters=2; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=2; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=2; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=3; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=3; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=3; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=4; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=4; total time=   0.0s


/home/haider/anaconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/haider/anaconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules

[CV] END ...............................kmeans__n_clusters=4; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=5; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=5; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=5; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=6; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=6; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=6; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=7; total time=   0.0s


/home/haider/anaconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/haider/anaconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules

[CV] END ...............................kmeans__n_clusters=7; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=7; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=8; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=8; total time=   0.0s
[CV] END ...............................kmeans__n_clusters=8; total time=   0.1s


...

[CV] END ..............................kmeans__n_clusters=97; total time=   0.1s
[CV] END ..............................kmeans__n_clusters=97; total time=   0.1s
[CV] END ..............................kmeans__n_clusters=97; total time=   0.1s
[CV] END ..............................kmeans__n_clusters=98; total time=   0.1s


/home/haider/anaconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/haider/anaconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules

[CV] END ..............................kmeans__n_clusters=98; total time=   0.1s
[CV] END ..............................kmeans__n_clusters=98; total time=   0.1s
[CV] END ..............................kmeans__n_clusters=99; total time=   0.1s
[CV] END ..............................kmeans__n_clusters=99; total time=   0.0s
[CV] END ..............................kmeans__n_clusters=99; total time=   0.0s


/home/haider/anaconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/haider/anaconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules

,estimator,Pipeline(step...egression())])
,param_grid,"{'kmeans__n_clusters': range(2, 100)}"
,scoring,None
,n_jobs,None
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_clusters,40


In [22]:
grid_clf.best_params_

{'kmeans__n_clusters': 40}

In [23]:
grid_clf.score(X_test, y_test)

0.9555555555555556

## DBSCAN

- This algorithm defines clusters as continuous regions of high density.
- For each instance, number of instances at a distance $<\epsilon$ are calculated (called $\epsilon$-_neighbourhood_)
- If an instance has atleast `min_samples` instances in the $\epsilon$-neighbourhood (including itself), then it is considered a core instance. In other words, core instances are those that are in dense regions.
- All instances in the neighbourhood of a core instance belong to the same cluster. This can include other core instances and therefore a long sequence of neighbouring core instances form a single cluster.
- Instance which is not a core instance is considered an anomaly.

In [2]:
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=1000, noise=0.05)
dbscan = DBSCAN(eps=0.05, min_samples=5)
dbscan.fit(X)

,eps,0.05
,min_samples,5
,metric,'euclidean'
,metric_params,None
,algorithm,'auto'
,leaf_size,30
,p,None
,n_jobs,None


In [3]:
dbscan.labels_

array([ 0,  1,  0,  0,  2,  0,  0,  3,  0,  2, -1,  0,  2,  0,  1,  4,  2,
        5,  6,  1, 10,  1,  0,  1,  4,  7,  6,  0,  0,  8,  2,  4, -1,  4,
        6,  8,  4,  6,  6,  1,  6,  0,  2,  0,  1,  2,  0,  0,  6, -1,  7,
        0,  8,  1,  5,  0,  0,  4,  4,  1,  9, 10,  0,  0,  9,  0,  8,  3,
        4,  0,  6, -1,  4,  4,  6,  2,  7, -1,  4, -1,  6,  2, -1,  4,  1,
        9,  7,  0,  1,  1, 10,  4,  0, -1,  9,  0,  0,  4, -1, -1,  1,  0,
        6, -1,  0,  2,  8,  7,  0,  8,  4,  2,  6, -1,  0,  0,  7, -1,  2,
        2,  9,  1,  6,  0,  1,  9,  4,  8,  0,  1,  0,  0,  6,  0,  0,  2,
        1,  3,  6,  6,  2,  0, -1,  2,  8,  1,  9,  6,  2,  0,  0,  0,  2,
        9,  4,  4,  1,  1,  4,  2,  6,  0,  4,  0,  4,  6,  0, -1,  9,  4,
        0,  0,  0,  0,  0,  6, 11,  7,  0,  0,  4,  7,  0,  2,  0,  6,  1,
        4,  6,  1,  1, -1,  4,  1,  0,  8, 10,  3,  4,  8,  5,  7,  0,  4,
        2,  0,  4,  2,  3,  8,  6,  0,  9, 11, -1,  2,  6,  6, 10,  0,  0,
        3,  0,  0,  2,  0

In [4]:
len(dbscan.core_sample_indices_) # indices of core instances

810

In [5]:
dbscan.core_sample_indices_

array([  0,   1,   2,   3,   4,   5,   7,   8,   9,  11,  12,  14,  15,
        16,  17,  18,  19,  22,  24,  25,  26,  27,  28,  29,  30,  31,
        33,  34,  35,  37,  38,  39,  40,  42,  43,  44,  46,  48,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  69,  70,  72,  73,  74,  75,  76,  78,  80,  81,  83,  84,
        85,  86,  87,  88,  90,  92,  94,  95,  96,  97, 100, 101, 102,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 114, 115, 116, 118,
       119, 120, 121, 122, 123, 124, 125, 126, 128, 129, 130, 131, 134,
       135, 136, 137, 138, 139, 140, 145, 148, 149, 150, 151, 152, 153,
       154, 155, 156, 157, 158, 160, 161, 162, 163, 164, 165, 166, 168,
       169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 183,
       184, 185, 186, 189, 190, 194, 195, 196, 197, 198, 199, 200, 202,
       203, 204, 205, 206, 207, 208, 209, 211, 212, 213, 215, 216, 217,
       218, 220, 221, 222, 223, 224, 225, 227, 228, 229, 230, 23

In [6]:
dbscan.components_

array([[-0.23815598,  0.96338431],
       [ 1.52395732, -0.25514504],
       [ 0.34823896,  0.93774181],
       ...,
       [ 0.50588393, -0.32473277],
       [ 0.2507353 ,  0.99794731],
       [ 1.89542579,  0.09861997]], shape=(810, 2))

**Note**: DBSCAN does not have a `predict()` method. The authors let users choose, as different classification algorithms are better suited for different tasks. 

In [7]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=50)
knn.fit(dbscan.components_, dbscan.labels_[dbscan.core_sample_indices_])

,n_neighbors,50
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [9]:
import numpy as np

X_new = np.array([[-0.5, 0], [0, 0.5], [1, -0.1], [2, 1]])
knn.predict(X_new)

array([9, 4, 0, 6])

In [10]:
knn.predict_proba(X_new)

array([[0.  , 0.  , 0.  , 0.  , 0.22, 0.  , 0.  , 0.04, 0.  , 0.74, 0.  ,
        0.  ],
       [0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ,
        0.  ],
       [0.76, 0.02, 0.04, 0.18, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ,
        0.  ],
       [0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  , 0.  , 0.  , 0.  , 0.  ,
        0.  ]])

In [11]:
y_dist, y_pred_idx = knn.kneighbors(X_new, n_neighbors=1)
y_pred = dbscan.labels_[dbscan.core_sample_indices_][y_pred_idx]
y_pred[y_dist > 0.2] = -1
y_pred.ravel()

array([-1,  4,  0, -1])

## Other Clustering Algorithms

1. **Agglomerative Clustering**: hierarchy of clusters is built from bottom up.
2. **Balanced Iterative Reducing and Clustering using Hierarchies (BIRCH)**: designed for very large datasets.
3. **Mean-Shift**: it starts by placing a cricle around each instance, then for each circle it computes the mean of all the instances within it and shift so that it is centered on the mean. This step is repeated until circles stop moving. All instances whose circles have settled close enough are assigned to the same cluster. It is not suited for large datasets.
4. **Affinity Propagation**: it uses a voting system, where instances vote for similar instances to be their representatives and once algorithm converges, each representative and its votes form a cluster. It is also not suited for large datasets.
5. **Spectral clustering**: This takes a similarity matrix between the instances and creates a low-dimensional embedding from it, then uses another clustering algorithm in this low-dimensional space. It is also not suited for large daasets, and also struggles when clusters have different sizes.